In [ ]:
# ============================================
# CELL 0 — Install core deps (run once per runtime)
# ============================================
!pip install -q pandas numpy pyarrow fastparquet spacy tqdm

# Small English model for the optional NLP step
#!python -m spacy download en_core_web_sm


In [ ]:
# ============================================
# CELL 1 — Mount Drive & define paths
# ============================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, html, unicodedata, hashlib
import pandas as pd
import numpy as np

# ---- BASE PATHS ----
BASE = "/content/drive/MyDrive/PropInsight"  # << change if needed

# RAW Reddit data (nested JSON/JSONL; can contain many files)
RAW_REDDIT_DIR = f"{BASE}/raw/reddit"

# Domain corpora
SINGLEX_CSV  = f"{BASE}/corpus/Singlish/lexicon.csv"
REGEX_JSONL  = f"{BASE}/corpus/SGPropertyDomain/regex_patterns.jsonl"
RULER_ORIG   = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.jsonl"
VOCAB_DIR    = f"{BASE}/corpus/SGPropertyDomain/vocab"  # *.txt category vocab
RULER_MERGED = f"{BASE}/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl"

# Outputs
OUTDIR       = f"{BASE}/preprocess/forum/reddit_2023_2025"
HASH_CHECKPT = f"{BASE}/labeled/checkpoints/reddit/reddit_hashes.parquet"

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
Path(Path(HASH_CHECKPT).parent).mkdir(parents=True, exist_ok=True)

print("BASE:", BASE)
print("RAW_REDDIT_DIR:", RAW_REDDIT_DIR)
print("OUTDIR:", OUTDIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/PropInsight
RAW_REDDIT_DIR: /content/drive/MyDrive/PropInsight/raw/reddit
OUTDIR: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_2023_2025


In [ ]:
# ============================================
# CELL 2 — Helpers (loading, cleaning, regex, Singlish)
# ============================================
import glob
from tqdm.auto import tqdm

def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()

def clean_basic(text: str) -> str:
    """Remove HTML/URLs/markdown links + normalize."""
    if not isinstance(text, str): return ""
    s = html.unescape(text)
    s = unicodedata.normalize("NFKC", s)
    # markdown links [text](url)
    s = re.sub(r"\[.*?\]\(https?://[^\s)]+\)", " ", s)
    # plain URLs / emails
    s = re.sub(r"https?://\S+|www\.\S+|\S+@\S+\.\S+", " ", s)
    # common deletion tokens
    s = re.sub(r"^\s*\[(deleted|removed)\]\s*$", " ", s, flags=re.I)
    # strip HTML tags
    s = re.sub(r"<[^>]+>", " ", s)
    # collapse punctuation & whitespace
    s = re.sub(r"([.,!?;:]){2,}", r"\1", s)
    return normalize_ws(s)

def make_hash(s: str) -> str:
    return hashlib.md5((s or "").lower().encode("utf-8")).hexdigest()

def read_json_any(path: Path):
    """Read .json or .jsonl into a Python list of posts (dicts)."""
    try:
        if path.suffix.lower() == ".jsonl":
            rows=[]
            with path.open("r", encoding="utf-8", errors="ignore") as f:
                for ln in f:
                    ln=ln.strip()
                    if ln:
                        try: rows.append(json.loads(ln))
                        except: pass
            return rows
        else:
            obj = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
            return obj if isinstance(obj, list) else [obj]
    except Exception as e:
        print("[WARN] Skipping unreadable file:", path, e)
        return []

def compile_regexes_from_jsonl(path: Path):
    """Load patterns from regex_patterns.jsonl -> list[(colname, compiled_rx)]."""
    pats=[]
    if not path.exists(): return pats
    with path.open("r", encoding="utf-8") as f:
        for ln in f:
            ln=ln.strip()
            if not ln: continue
            try:
                d=json.loads(ln)
                pat=d.get("pattern")
                name=d.get("name","pattern")
                if pat:
                    pats.append((f"rx_{name}", re.compile(pat, re.I)))
            except Exception:
                pass
    return pats

def build_singlish_dict(csv_path: str):
    """Load Singlish lexicon -> set(words) and optional description map."""
    df = pd.read_csv(csv_path)
    df.columns = [c.strip().lower() for c in df.columns]
    if "word" not in df.columns:
        raise ValueError(f"'word' column missing in {csv_path}. Found: {df.columns.tolist()}")
    df = df.dropna(subset=["word"])
    words_set = set(df["word"].astype(str).str.strip().str.lower())
    meta_map = {}
    if "description" in df.columns:
        for _, r in df.iterrows():
            w = str(r["word"]).strip().lower()
            meta_map[w] = {"description": str(r["description"]).strip()} if pd.notna(r.get("description")) else {}
    return words_set, meta_map

def find_singlish_terms(text: str, words_set):
    toks = re.findall(r"[A-Za-z][A-Za-z\-']+", text or "")
    found = sorted(set([t.lower() for t in toks if t.lower() in words_set]))
    return found


In [ ]:
# ============================================
# CELL 3 — Merge vocab/*.txt into EntityRuler patterns (no dups)
# ============================================
def merge_vocab_to_entityruler(vocab_dir: str, existing_jsonl: str, merged_out: str):
    p_voc = Path(vocab_dir)
    p_ex  = Path(existing_jsonl)
    p_out = Path(merged_out)
    # read existing
    existing=[]
    if p_ex.exists():
        with p_ex.open("r", encoding="utf-8") as f:
            for ln in f:
                ln=ln.strip()
                if ln:
                    try: existing.append(json.loads(ln))
                    except: pass

    def phrase_to_token_pattern(phrase: str):
        phrase = re.sub(r"\s+", " ", phrase).strip()
        if not phrase: return None
        return [{"LOWER": t.lower()} for t in phrase.split(" ") if t]

    def lowers_from_pattern(pat):
        if isinstance(pat, str):
            return tuple(re.sub(r"\s+"," ",pat).lower().split(" "))
        if isinstance(pat, dict):
            return (str(pat.get("LOWER", pat.get("TEXT",""))).lower(),)
        if isinstance(pat, list):
            outs=[]
            for tok in pat:
                if isinstance(tok, dict):
                    outs.append(str(tok.get("LOWER", tok.get("TEXT",""))).lower())
                else:
                    outs.append(str(tok).lower())
            return tuple(outs)
        return (str(pat).lower(),)

    def pat_key(rec):
        return (rec.get("label",""), lowers_from_pattern(rec.get("pattern","")))

    merged=[]; seen=set()
    for rec in existing:
        k = pat_key(rec)
        if k not in seen:
            seen.add(k); merged.append(rec)

    # add from vocab/*.txt
    if p_voc.exists():
        for txt in sorted(p_voc.glob("*.txt")):
            label = re.sub(r"[^A-Za-z0-9]+","_", txt.stem).strip("_").upper() or "DOMAIN"
            for raw in txt.read_text(encoding="utf-8", errors="ignore").splitlines():
                term = raw.strip()
                if not term: continue
                pat = phrase_to_token_pattern(term)
                if not pat: continue
                rec={"label":label,"pattern":pat,"id":term}
                k=pat_key(rec)
                if k not in seen:
                    seen.add(k); merged.append(rec)

    p_out.parent.mkdir(parents=True, exist_ok=True)
    with p_out.open("w", encoding="utf-8") as f:
        for rec in merged:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"[EntityRuler] merged → {p_out} (rules: {len(merged)})")
    return str(p_out)

# Run merge
ENTITYRULER_MERGED = merge_vocab_to_entityruler(VOCAB_DIR, RULER_ORIG, RULER_MERGED)


[EntityRuler] merged → /content/drive/MyDrive/PropInsight/corpus/SGPropertyDomain/spacy_entityruler_patterns.merged.jsonl (rules: 2158)


In [ ]:
# ============================================
# CELL 4 — Flatten Reddit nested JSON → DataFrame
# ============================================
def flatten_reddit_folder(raw_dir: str):
    files = sorted(glob.glob(f"{raw_dir}/**/*.json*", recursive=True))
    if not files:
        print("[WARN] No JSON/JSONL found in:", raw_dir)
        return pd.DataFrame()

    recs=[]
    for fp in tqdm(files, desc="Loading Reddit JSON"):
        posts = read_json_any(Path(fp))
        for post in posts:
            base = {
                "post_id": post.get("post_id"),
                "subreddit": post.get("subreddit"),
                "title": post.get("title",""),
                "selftext": post.get("selftext",""),
                "author": post.get("author"),
                "score_post": post.get("score"),
                "num_comments": post.get("num_comments"),
                "url": post.get("url"),
                "permalink": post.get("permalink"),
                "created_utc_post": post.get("created_utc"),
            }
            # include post as a row (treat selftext as main body if no comments)
            recs.append({**base,
                         "comment_id": None,
                         "comment_author": post.get("author"),
                         "comment_body": post.get("selftext",""),
                         "comment_score": None,
                         "created_utc_comment": None})

            # include each comment
            for c in post.get("comments", []) or []:
                recs.append({**base,
                             "comment_id": c.get("comment_id"),
                             "comment_author": c.get("comment_author"),
                             "comment_body": c.get("comment_body"),
                             "comment_score": c.get("comment_score"),
                             "created_utc_comment": c.get("created_utc")})
    df = pd.DataFrame(recs)
    return df

df = flatten_reddit_folder(RAW_REDDIT_DIR)

# Optional: if you also want to add one-off local file (uploaded to Colab):
# Try merging /mnt/data file if exists:
local_test = Path("/mnt/data/reddit_property_nested_sgpropinvesting.json")
if local_test.exists():
    print("Merging local uploaded JSON:", local_test)
    extra = pd.DataFrame(flatten_reddit_folder(str(local_test.parent)))
    # (the function will try to read the whole folder; this is OK for a single file dir)
    df = pd.concat([df, extra], ignore_index=True, sort=False)

print("Raw flattened rows:", len(df))
df.head(2)


Loading Reddit JSON:   0%|          | 0/2 [00:00<?, ?it/s]

Raw flattened rows: 99403


,post_id,subreddit,title,selftext,author,score_post,num_comments,url,permalink,created_utc_post,comment_id,comment_author,comment_body,comment_score,created_utc_comment
0,1o050p5,singapore,Newlyweds host buffet spread along Tampines HD...,,Powerful_Office3936,248,52,https://mothership.sg/2025/10/just-married-buf...,https://reddit.com/r/singapore/comments/1o050p...,2025-10-07T12:20:10,None,Powerful_Office3936,,NaN,None
1,1o050p5,singapore,Newlyweds host buffet spread along Tampines HD...,,Powerful_Office3936,248,52,https://mothership.sg/2025/10/just-married-buf...,https://reddit.com/r/singapore/comments/1o050p...,2025-10-07T12:20:10,ni73slb,KopiSiewSiewDai,Reminder to all that you can do anything you w...,279.0,2025-10-07T12:59:40


In [ ]:
# ============================================
# CELL 5 — Timestamp → SG time; Filter 2023–2025; Build 'body'; Clean; Low-signal filter
# ============================================
# Choose comment timestamp when present, else post timestamp
df["date"] = pd.to_datetime(
    df["created_utc_comment"].fillna(df["created_utc_post"]),
    errors="coerce", utc=True
)

# Convert to Singapore time (optional, but nice for dashboards)
df["date"] = df["date"].dt.tz_convert("Asia/Singapore")

# Keep only 2023–2025
mask_window = (df["date"] >= "2023-01-01") & (df["date"] < "2026-01-01")
before = len(df)
df = df[mask_window].copy()
print(f"[Date Filter] Kept {len(df)} / {before} rows (2023–2025)")

# Build body from title, selftext, comment
df["title"]       = df.get("title","").astype(str)
df["selftext"]    = df.get("selftext","").astype(str)
df["comment_body"]= df.get("comment_body","").astype(str)

df["body_raw"] = (df["title"].fillna("") + "\n\n" +
                  df["selftext"].fillna("") + "\n\n" +
                  df["comment_body"].fillna("")).str.strip()
df["body"] = df["body_raw"].apply(clean_basic)

# Drop low-signal rows
n0 = len(df)
df = df[df["body"].str.len() >= 60].copy()
print(f"[Low-signal] Removed {n0-len(df)} short rows")

# Optional: drop bodies without alphabetic characters
n0 = len(df)
df = df[df["body"].str.contains(r"[A-Za-z]", na=False)].copy()
print(f"[Alpha-check] Removed {n0-len(df)} rows with no alphabetic characters")

# Add temporal partitions
df["year"]    = df["date"].dt.year
df["month"]   = df["date"].dt.to_period("M").astype(str)
df["quarter"] = df["date"].dt.to_period("Q").astype(str)


[Date Filter] Kept 90194 / 99403 rows (2023–2025)
[Low-signal] Removed 213 short rows
[Alpha-check] Removed 0 rows with no alphabetic characters


/tmp/ipython-input-1068241406.py:41: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"]   = df["date"].dt.to_period("M").astype(str)
/tmp/ipython-input-1068241406.py:42: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["quarter"] = df["date"].dt.to_period("Q").astype(str)


In [ ]:
# ============================================
# CELL 6 — Dedup (early key + content hash) + Global hash checkpoint
# ============================================
# Early dedup: based on URL + title + first chunk of selftext
df["_k1"] = (
    df.get("url","").astype(str).str.lower() + "||" +
    df["title"].astype(str).str.lower() + "||" +
    df["selftext"].astype(str).str.lower().str[:300]
)
before = len(df)
df = df.drop_duplicates("_k1").drop(columns=["_k1"])
print(f"[Dedupe-early] Removed {before-len(df)} rows")

# Content hash
df["hash"] = df["body"].str.lower().map(lambda s: hashlib.md5(s.encode("utf-8")).hexdigest())
before = len(df)
df = df.drop_duplicates("hash").copy()
print(f"[Dedupe-hash] Removed {before-len(df)} rows; kept {len(df)}")

# Global hash store (so future runs skip relabeling)
try:
    seen = pd.read_parquet(HASH_CHECKPT)
    seen_set = set(seen["hash"].astype(str))
except Exception:
    seen_set = set()

new_count = (~df["hash"].isin(seen_set)).sum()
print(f"New unique bodies (not in global hash store): {new_count}")

# Do NOT exclude already-seen here; keep all for preprocess.
# The hash store is mostly for labeling to skip repeats across runs.


[Dedupe-early] Removed 86695 rows
[Dedupe-hash] Removed 11 rows; kept 3275
New unique bodies (not in global hash store): 3275


In [ ]:
# ============================================
# CELL 7 — Enrichment: regex flags + Singlish + EntityRuler
# ============================================
# 7a) Regex flags
pats = compile_regexes_from_jsonl(Path(REGEX_JSONL))
if pats:
    for col, rx in pats:
        try:
            df[col] = df["body"].str.contains(rx, na=False)
        except Exception:
            df[col] = False
    print(f"[Regex] Added {len(pats)} rx_* columns")
else:
    print("[Regex] No patterns found or file missing; skipping")

# 7b) Singlish
try:
    sing_words, sing_meta = build_singlish_dict(SINGLEX_CSV)
    df["singlish_terms"] = df["body"].apply(lambda s: find_singlish_terms(s, sing_words))
    df["has_singlish"]   = df["singlish_terms"].str.len().gt(0)
    print(f"[Singlish] Terms detected in {df['has_singlish'].sum()} rows")
except Exception as e:
    print("[Singlish] Skipped:", e)
    df["singlish_terms"] = [[] for _ in range(len(df))]
    df["has_singlish"]   = False

# 7c) EntityRuler (domain entities)
try:
    import spacy
    nlp = spacy.blank("en")
    ruler = nlp.add_pipe("entity_ruler")
    ruler.from_disk(RULER_MERGED)
    ents=[]
    for doc in nlp.pipe(df["body"].astype(str).tolist(), batch_size=64):
        ents.append([{"text": e.text, "label": e.label_} for e in doc.ents])
    df["entities"] = ents
    print(f"[EntityRuler] Entities attached for {len(df)} rows")
except Exception as e:
    print("[EntityRuler] Skipped:", e)
    df["entities"] = [[] for _ in range(len(df))]


[Regex] No patterns found or file missing; skipping
[Singlish] Terms detected in 2974 rows
[EntityRuler] Entities attached for 3275 rows


In [ ]:
# ============================================
# CELL 8 — Save main clean corpus (CSV/Parquet)
# ============================================
core_cols = [
    "hash","date","year","quarter","month",
    "subreddit","post_id","comment_id","author","comment_author",
    "title","selftext","comment_body","body",
    "url","permalink","score_post","comment_score","num_comments"
]
# add enrichment cols at the end
rx_cols  = [c for c in df.columns if c.startswith("rx_")]
extra_cols = ["has_singlish","singlish_terms","entities"]

df_out = df[ [c for c in core_cols if c in df.columns] + rx_cols + extra_cols ].copy()

csv_path = f"{OUTDIR}/reddit_clean_2023_2025.csv"
parq_path= f"{OUTDIR}/reddit_clean_2023_2025.parquet"
df_out.to_csv(csv_path, index=False)
try:
    df_out.to_parquet(parq_path, index=False)
except Exception as e:
    print("[WARN] Parquet save failed:", e)

print("Saved CSV:", csv_path)
print("Saved Parquet:", parq_path)
print("Rows saved:", len(df_out))


Saved CSV: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_2023_2025/reddit_clean_2023_2025.csv
Saved Parquet: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_2023_2025/reddit_clean_2023_2025.parquet
Rows saved: 3275


In [ ]:
# ============================================
# CELL 9 — OPTIONAL NLP enrichment (tokens/lemmas/POS/NER + sentence table)
# ============================================
RUN_NLP_ENRICHMENT = True  # flip to True to run

if RUN_NLP_ENRICHMENT:
    import spacy
    try:
        nlp2 = spacy.load("en_core_web_sm", exclude=[])
    except Exception:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=False)
        nlp2 = spacy.load("en_core_web_sm", exclude=[])

    # Re-attach EntityRuler before 'ner' to get both domain + model entities in docs
    try:
        er = nlp2.add_pipe("entity_ruler", before="ner")
        er.from_disk(RULER_MERGED)
    except Exception as e:
        print("[WARN] Could not attach EntityRuler to full pipeline:", e)

    texts = df_out["body"].astype(str).tolist()
    docs = list(nlp2.pipe(texts, batch_size=64, n_process=2))

    df_out["tokens"] = [[t.text for t in d] for d in docs]
    df_out["lemmas"] = [[t.lemma_ for t in d] for d in docs]
    df_out["pos"]    = [[t.pos_ for t in d] for d in docs]
    df_out["deps"]   = [[t.dep_ for t in d] for d in docs]
    df_out["entities_ner"] = [[{"text": e.text, "label": e.label_} for e in d.ents] for d in docs]
    df_out["aspect_candidates"] = [[nc.text for nc in d.noun_chunks] for d in docs]

    enr_dir = Path(OUTDIR) / "nlp_enriched"
    enr_dir.mkdir(parents=True, exist_ok=True)
    df_out.to_parquet(enr_dir / "reddit_enriched+nlp.parquet", index=False)
    print("Saved:", enr_dir / "reddit_enriched+nlp.parquet")

    # Sentence table
    sent_rows=[]
    for i, d in enumerate(docs):
        for j, s in enumerate(d.sents):
            sent_rows.append({
                "doc_id": i, "sent_id": j, "text": s.text,
                "tokens": [t.text for t in s],
                "lemmas": [t.lemma_ for t in s],
                "pos":    [t.pos_ for t in s],
                "deps":   [t.dep_ for t in s],
                "date":   df_out.iloc[i]["date"]
            })
    pd.DataFrame(sent_rows).to_parquet(enr_dir / "reddit_sentences.parquet", index=False)
    print("Saved:", enr_dir / "reddit_sentences.parquet")
else:
    print("[INFO] RUN_NLP_ENRICHMENT=False → skipping spaCy heavy step.")


Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_2023_2025/nlp_enriched/reddit_enriched+nlp.parquet
Saved: /content/drive/MyDrive/PropInsight/preprocess/forum/reddit_2023_2025/nlp_enriched/reddit_sentences.parquet


In [ ]:
# ============================================
# CELL 10 — (Optional) Update global hash checkpoint now
# ============================================
try:
    cur = pd.read_parquet(HASH_CHECKPT)
except Exception:
    cur = pd.DataFrame(columns=["hash"])

to_add = df_out[["hash"]].drop_duplicates()
merged = pd.concat([cur, to_add], ignore_index=True).drop_duplicates("hash")
merged.to_parquet(HASH_CHECKPT, index=False)
print("Updated hash checkpoint:", HASH_CHECKPT, "| total hashes:", len(merged))


Updated hash checkpoint: /content/drive/MyDrive/PropInsight/labeled/checkpoints/reddit/reddit_hashes.parquet | total hashes: 3275
